# Denoising and Faster Whisper CPU comparison

Compare the **same short excerpt** before and after `noisereduce`. This notebook extracts audio from a local video or audio file and does not run any transcription or GPU model. Results go to Git-ignored `runs/denoise_lab/`. The source is never modified.

Use the repository's `.venv-moss` Python/kernel after `python setup_env.py moss --prep-only` (or another kernel with `requirements-prep.txt`). For the optional CPU transcript comparison, install `python -m pip install -r requirements-denoise-asr.txt` in that same kernel. Put `input_file` in ignored `local_config.json`, or set `INPUT_FILE` below. **Clear notebook outputs before committing**: audio players and transcripts contain recording data.

[noisereduce documentation](https://github.com/timsainb/noisereduce/blob/master/README.md): `prop_decrease` controls how much estimated noise is suppressed. The stationary option can use a short, speech-free noise sample.


## 1. Choose a file and listening window

Edit the numbers here, then **Run All**. Start with a 20-second excerpt that contains both noise and quiet speech. `STRENGTHS` gives separate outputs at the same timestamps; higher values can remove speech as well as noise. To try stationary gating, set `NOISE_ONLY_START_SECONDS` to the start of a speech-free section in the **same source**.


In [ ]:
from pathlib import Path
import json

INPUT_FILE = ""  # local absolute path; blank uses ignored local_config.json
AUDIO_STREAM = 0  # first audio track of a video
START_SECONDS = 0  # choose a window where you can hear speech
DURATION_SECONDS = 20
STRENGTHS = (0.45, 0.75, 1.0)  # noisereduce prop_decrease, 0..1
NOISE_ONLY_START_SECONDS = None  # e.g. 5; None skips stationary gating
NOISE_ONLY_DURATION_SECONDS = 3
TIME_CONSTANT_S = 2.0  # nonstationary noise tracking; seconds
N_STD_THRESH_STATIONARY = 1.5  # stationary gate sensitivity
REMOVED_VARIANT_INDEX = 2  # 1=first denoised output, 2=second, etc.
RUN_CPU_ASR = True  # set False to listen without downloading a Whisper model
ASR_MODEL = 'base.en'  # CPU-friendly; try 'small.en' for a stronger, slower English model
ASR_LANGUAGE = 'en'
ASR_BEAM_SIZE = 5
ASR_CPU_THREADS = 4

if not INPUT_FILE and Path('local_config.json').is_file():
    INPUT_FILE = json.loads(Path('local_config.json').read_text(encoding='utf-8')).get('input_file', '')
if not INPUT_FILE:
    raise ValueError('Set INPUT_FILE or create local_config.json with an input_file path')
source_path = Path(INPUT_FILE).expanduser().resolve()
if not source_path.is_file():
    raise FileNotFoundError(source_path)
if AUDIO_STREAM < 0 or START_SECONDS < 0 or not 1 <= DURATION_SECONDS <= 60:
    raise ValueError('AUDIO_STREAM and START_SECONDS must be nonnegative; DURATION_SECONDS must be 1..60')
if not STRENGTHS or any(not 0 < value <= 1 for value in STRENGTHS):
    raise ValueError('Choose one or more STRENGTHS in (0, 1]')
if NOISE_ONLY_START_SECONDS is not None and (NOISE_ONLY_START_SECONDS < 0 or NOISE_ONLY_DURATION_SECONDS <= 0):
    raise ValueError('The noise-only window needs a nonnegative start and positive duration')
print('Input:', source_path.name, '| preview:', START_SECONDS, 'to', START_SECONDS + DURATION_SECONDS, 's')


## 2. Extract the same audio excerpt for every setting

FFmpeg selects the requested audio track and creates 16 kHz mono PCM WAV. This mirrors the current transcription notebooks' input format; it does not preserve multichannel spatial cues.


In [ ]:
import re
import subprocess
import wave
from datetime import datetime, timezone

import numpy as np
from pipeline_common import ffmpeg_tools

ffmpeg, _ = ffmpeg_tools()
safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', source_path.stem).strip('_')[:40] or 'audio'
run_dir = Path('runs/denoise_lab').resolve() / (safe_name + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
run_dir.mkdir(parents=True, exist_ok=False)

def extract_excerpt(start_seconds, duration_seconds, destination):
    command = [ffmpeg, '-hide_banner', '-loglevel', 'error', '-nostdin', '-y',
               '-ss', str(start_seconds), '-i', str(source_path), '-t', str(duration_seconds),
               '-map', f'0:a:{AUDIO_STREAM}', '-vn', '-ac', '1', '-ar', '16000',
               '-c:a', 'pcm_s16le', str(destination)]
    subprocess.run(command, check=True)

def read_pcm16(path):
    with wave.open(str(path), 'rb') as handle:
        if (handle.getframerate(), handle.getnchannels(), handle.getsampwidth()) != (16000, 1, 2):
            raise ValueError(f'Unexpected WAV format: {path}')
        samples = np.frombuffer(handle.readframes(handle.getnframes()), dtype='<i2')
    if samples.size == 0:
        raise ValueError(f'Empty excerpt: {path}; check the chosen start time')
    return samples.astype(np.float32) / 32768.0

original_path = run_dir / 'original.wav'
extract_excerpt(START_SECONDS, DURATION_SECONDS, original_path)
original = read_pcm16(original_path)
noise_only = None
if NOISE_ONLY_START_SECONDS is not None:
    noise_path = run_dir / 'noise_only_reference.wav'
    extract_excerpt(NOISE_ONLY_START_SECONDS, NOISE_ONLY_DURATION_SECONDS, noise_path)
    noise_only = read_pcm16(noise_path)
print(f'Extracted {len(original) / 16000:.2f} s to {run_dir}')


## 3. Apply adjustable spectral gating

Nonstationary gating estimates a changing noise floor from the excerpt. Stationary gating uses the optional speech-free reference. All outputs retain the original duration and sampling rate.


In [ ]:
import noisereduce as nr

def save_pcm16(path, samples):
    pcm = np.rint(np.clip(samples, -1.0, 32767 / 32768) * 32768).astype('<i2')
    with wave.open(str(path), 'wb') as handle:
        handle.setnchannels(1)
        handle.setsampwidth(2)
        handle.setframerate(16000)
        handle.writeframes(pcm.tobytes())

variants = [('Original', original, original_path)]
for strength in STRENGTHS:
    label = f'Nonstationary {strength:.2f}'
    processed = nr.reduce_noise(y=original, sr=16000, stationary=False,
                                prop_decrease=strength, time_constant_s=TIME_CONSTANT_S,
                                n_fft=512, n_jobs=1).astype(np.float32)
    path = run_dir / f'nonstationary_{strength:.2f}.wav'
    save_pcm16(path, processed)
    variants.append((label, read_pcm16(path), path))
    if noise_only is not None:
        label = f'Stationary {strength:.2f}'
        processed = nr.reduce_noise(y=original, sr=16000, y_noise=noise_only,
                                    stationary=True, prop_decrease=strength,
                                    n_std_thresh_stationary=N_STD_THRESH_STATIONARY,
                                    n_fft=512, n_jobs=1).astype(np.float32)
        path = run_dir / f'stationary_{strength:.2f}.wav'
        save_pcm16(path, processed)
        variants.append((label, read_pcm16(path), path))

if any(len(samples) != len(original) for _, samples, _ in variants):
    raise ValueError('A denoised result changed the excerpt length')
settings = {'source_path': str(source_path), 'audio_stream': AUDIO_STREAM,
            'start_seconds': START_SECONDS, 'duration_seconds': DURATION_SECONDS,
            'strengths': STRENGTHS, 'noise_only_start_seconds': NOISE_ONLY_START_SECONDS,
            'noise_only_duration_seconds': NOISE_ONLY_DURATION_SECONDS,
            'time_constant_s': TIME_CONSTANT_S,
            'n_std_thresh_stationary': N_STD_THRESH_STATIONARY}
(run_dir / 'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
print('Saved', len(variants), 'versions in', run_dir)


## 4. Listen at unchanged playback gain

The players use the same excerpt and `normalize=False`, so quieter playback is not automatically boosted. Listen for both less background noise **and** preservation of soft words, consonants and overlapping speech. The RMS column describes signal level, not speech quality.


In [ ]:
import html
from IPython.display import Audio, HTML, display

rows = []
for label, samples, path in variants:
    rms = float(np.sqrt(np.mean(np.square(samples.astype(np.float64)))))
    rms_dbfs = 20 * np.log10(max(rms, 1e-12))
    rows.append(f'<tr><td>{html.escape(label)}</td><td>{rms_dbfs:.1f}</td><td>{html.escape(path.name)}</td></tr>')
display(HTML('<table><tr><th>Version</th><th>RMS dBFS</th><th>WAV file</th></tr>' + ''.join(rows) + '</table>'))
for label, samples, _ in variants:
    display(HTML(f'<h4>{html.escape(label)}</h4>'))
    display(Audio(data=samples, rate=16000, normalize=False))

if 0 < REMOVED_VARIANT_INDEX < len(variants):
    label, processed, _ = variants[REMOVED_VARIANT_INDEX]
    removed = np.clip(original - processed, -1.0, 1.0)
    display(HTML(f'<h4>Removed sound: Original − {html.escape(label)}</h4>'))
    display(Audio(data=removed, rate=16000, normalize=False))
print('WAV files and settings:', run_dir)


## 5. Compare Faster Whisper transcripts on CPU

This is **ASR only**, with no speaker diarization. Every version uses the same model, beam size, language and `vad_filter=False`, so only the audio changes. The first run downloads the selected public model. Timestamps below are relative to the 20-second excerpt. Transcripts are saved only in Git-ignored `runs/denoise_lab/`. Without a manually checked reference transcript, differences cannot establish which version is more accurate.


In [ ]:
if RUN_CPU_ASR:
    try:
        from faster_whisper import WhisperModel
    except ImportError as exc:
        raise RuntimeError('Install the optional CPU ASR dependencies in this kernel: python -m pip install -r requirements-denoise-asr.txt') from exc
    import time
    import html
    from IPython.display import HTML, display
    model = WhisperModel(ASR_MODEL, device='cpu', compute_type='int8', cpu_threads=ASR_CPU_THREADS)
    asr_results = []
    for label, samples, path in variants:
        started = time.perf_counter()
        segment_stream, info = model.transcribe(samples, language=ASR_LANGUAGE,
                                               beam_size=ASR_BEAM_SIZE, vad_filter=False,
                                               condition_on_previous_text=False)
        segments = list(segment_stream)  # Faster Whisper does the work when iterated
        record = {'version': label, 'file': path.name,
                  'elapsed_seconds': round(time.perf_counter() - started, 2),
                  'segments': [{'start': round(s.start, 2), 'end': round(s.end, 2),
                                'text': s.text.strip()} for s in segments]}
        asr_results.append(record)
        print(f"{label}: {len(segments)} segment(s), {record['elapsed_seconds']:.1f} s CPU")
    payload = {'model': ASR_MODEL, 'language': ASR_LANGUAGE, 'beam_size': ASR_BEAM_SIZE,
               'compute_type': 'int8', 'vad_filter': False, 'results': asr_results}
    (run_dir / 'asr_comparison.json').write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    table_rows = []
    for result in asr_results:
        transcript = '\n'.join(
            f"[{s['start']:.1f}–{s['end']:.1f}s] {s['text']}" for s in result['segments']
        ) or '(No speech detected)'
        table_rows.append('<tr><th style="vertical-align:top;text-align:left;padding:8px">'
                          + html.escape(result['version']) + '</th><td style="padding:8px;white-space:pre-wrap">'
                          + html.escape(transcript) + '</td></tr>')
    display(HTML('<table style="width:100%;border-collapse:collapse"><tr><th>Audio version</th>'
                 '<th>Faster Whisper transcript</th></tr>' + ''.join(table_rows) + '</table>'))
    if not asr_results[0]['segments']:
        print('Original has no detected speech. Choose a window with audible speech before judging denoising; text appearing only after denoising may be a hallucination.')
else:
    print('CPU ASR skipped. Set RUN_CPU_ASR=True to compare transcripts.')


## What to listen for

Pick a setting only after listening to the **same** excerpt repeatedly. If words appear in the *Removed sound* player, that setting is removing speech. Try a quieter or more representative noise-only reference for stationary gating, or lower `STRENGTHS`. Speech from another person is not ordinary stationary noise, so this filter will not reliably separate overlapping clinicians. Results in `runs/denoise_lab/` are ignored by Git. Clear notebook outputs before sharing this notebook.
